In [1]:
from sqlalchemy import text
from app.db.session import session_scope
with session_scope() as s:
    v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
    if v:
        print(f'vector 확장 : 이미 켜져 있습니다 — {v}')
    else:
        s.execute(text('CREATE EXTENSION IF NOT EXISTS vector'))
        s.commit()
        v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
        print(f'vector 확장 : 켰습니다 — {v}')


vector 확장 : 이미 켜져 있습니다 — 0.8.6


In [2]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [3]:
from app.models.document import Chunk
from app.db.session import session_scope
from app.models.document import Chunk
from sqlalchemy import func, select

queries = ['숙박', '숙박비', '출장 갈 때 숙박비 얼마?', '출장 숙박비']
zeros = []
with session_scope() as s:
    for q in queries:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{q}%'))) or 0
        print(f'{q!r:<28} → {n}줄')
        if n == 0:
            zeros.append(q)
print()
print(f'넷 중 {len(zeros)}개가 0 줄입니다.')


'숙박'                         → 5줄
'숙박비'                        → 5줄
'출장 갈 때 숙박비 얼마?'             → 0줄
'출장 숙박비'                     → 0줄

넷 중 2개가 0 줄입니다.


In [4]:
import importlib
from pathlib import Path
import app.models.base
importlib.reload(app.models.base)

from app.core.config import get_settings
get_settings.cache_clear()
settings = get_settings()

print('models/base.py 의 EMBED_DIM :', app.models.base.EMBED_DIM)
print('settings.embed_dim          :', settings.embed_dim)
print('settings.embed_provider     :', settings.embed_provider)
print('settings.embed_model_dir    :', settings.embed_model_dir)
print('settings.upstage_embed_model:', settings.upstage_embed_model)
print('.env :', f'{len(Path('.env').read_text(encoding='utf-8').splitlines())}줄')


models/base.py 의 EMBED_DIM : 1024
settings.embed_dim          : 1024
settings.embed_provider     : local
settings.embed_model_dir    : C:/hw/hw/python_basic/models/bge-m3
settings.upstage_embed_model: 
.env : 68줄


In [5]:
import importlib
import inspect
from app.services import document_service

importlib.reload(document_service)

job_src = inspect.getsource(document_service.run_ingest_job)
for needle in ['range(5)', 'steps[4]["time"]', 'int(5 / len(steps)']:
    print(f'{needle:<20}: ' + ('있음' if needle in job_src else '없음'))
path = next(SAMPLES.glob('DOC-FI-009_*_v1.4.docx')).as_posix()

job_id = document_service.start_ingest_job(doc_id='DOC-FI-009', version='v2.0', path=path)   # DB에 저장된 문서 버전 확인 
document_service.run_ingest_job(job_id)
job = document_service.get_job(job_id)
print()

print(f'status={job['status']} progress={job['progress']} chunk_count={job['chunk_count']}')
if job['status'] == '실패':
    print('message:', job['message'])
for step in job['steps']:
    print(f'  {step['state']:<5} {step['name']}')

range(5)            : 있음
steps[4]["time"]    : 없음
int(5 / len(steps)  : 있음

status=실패 progress=0 chunk_count=0
message: 문서 버전을 찾을 수 없습니다 :DOC-FI-009 v2.0
  todo  파일 검증
  todo  문서 파싱
  todo  표 -> 마크다운 변환
  todo  청킹
  todo  임베딩
  todo  검색 반영 및 현행 버전 지정


In [6]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.document import Chunk
with session_scope() as s:
    base = s.execute(select(Chunk).where(Chunk.embedding.is_not(None)).order_by(Chunk.id).limit(1)).scalar_one()
    base_vec = list(base.embedding)
    print(f'기준 청크 : {base.locator}   (벡터 길이 {len(base_vec)})')
    OPS = {'<=>  코사인 거리': Chunk.embedding.cosine_distance, '<->  L2 거리': Chunk.embedding.l2_distance, '<#>  내적의 음수': Chunk.embedding.max_inner_product}
    for name, op in OPS.items():
        print(f'\n{name}')
        stmt = select(Chunk.locator, op(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
        for rank, (loc, d) in enumerate(s.execute(stmt), 1):
            print(f'  {rank}. {loc:<26} {round(float(d), 4)}')


기준 청크 : DOC-FI-009_v2.1   (벡터 길이 1024)

<=>  코사인 거리
  1. DOC-FI-009_v2.1            0.0
  2. 제1장 제3조(발급)                0.3074
  3. 제2장 제4조(사용 범위) · 별표1       0.3164
  4. 제1장 제2조(적용 범위)             0.3389
  5. 제1장 제1조(목적)                0.3454

<->  L2 거리
  1. DOC-FI-009_v2.1            0.0
  2. 제1장 제3조(발급)                0.7841
  3. 제2장 제4조(사용 범위) · 별표1       0.7954
  4. 제1장 제2조(적용 범위)             0.8233
  5. 제1장 제1조(목적)                0.8311

<#>  내적의 음수
  1. DOC-FI-009_v2.1            -1.0
  2. 제1장 제3조(발급)                -0.6926
  3. 제2장 제4조(사용 범위) · 별표1       -0.6836
  4. 제1장 제2조(적용 범위)             -0.6611
  5. 제1장 제1조(목적)                -0.6546


In [7]:
with session_scope() as s:
    stmt = select(Chunk.locator, Chunk.embedding.cosine_distance(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
    rows = [(loc, round(float(d), 4), round(1.0 - float(d), 4)) for loc, d in s.execute(stmt)]
by_distance = rows
by_score = sorted(rows, key=lambda r: r[2], reverse=True)
print('[ distance 오름차순 ]        [ score 내림차순 ]')
for a, b in zip(by_distance, by_score):
    print(f'  {a[1]:<8}{a[0]:<22} {b[2]:<8}{b[0]}')
print('\n두 표의 순서가 같은가 :', [r[0] for r in by_distance] == [r[0] for r in by_score])


[ distance 오름차순 ]        [ score 내림차순 ]
  0.0     DOC-FI-009_v2.1        1.0     DOC-FI-009_v2.1
  0.3074  제1장 제3조(발급)            0.6926  제1장 제3조(발급)
  0.3164  제2장 제4조(사용 범위) · 별표1   0.6836  제2장 제4조(사용 범위) · 별표1
  0.3389  제1장 제2조(적용 범위)         0.6611  제1장 제2조(적용 범위)
  0.3454  제1장 제1조(목적)            0.6546  제1장 제1조(목적)

두 표의 순서가 같은가 : True
